In [1]:
# Import và đọc file đã làm sạch
import sqlite3
import pandas as pd

df_sf = pd.read_csv(
    "SF_Tree_Maintenance_Daily_Data_Cleaned.csv",
    parse_dates=["date"]
)

print(df_sf.shape)


(273224, 10)


In [2]:
# Tạo database và bảng demand
conn = sqlite3.connect("sf_tree_demand.db")
cursor = conn.cursor()

df_sf.to_sql(
    "demand",
    conn,
    if_exists="replace",
    index=False
)

print("Đã tạo bảng SQL: demand")


Đã tạo bảng SQL: demand


In [3]:
# Kiểm tra bảng đã tạo
pd.read_sql_query("""
    SELECT *
    FROM demand
    LIMIT 5
""", conn)


,nbhd,date,n,permits_active,PRCP,AWND,TMAX,year,month,dow
0,Bayview Hunters Point,2008-07-01 00:00:00,0,0,0.0,4.6,19.4,2008,7,1
1,Bayview Hunters Point,2008-07-02 00:00:00,0,0,0.0,4.6,19.4,2008,7,2
2,Bayview Hunters Point,2008-07-03 00:00:00,0,0,0.0,4.6,19.4,2008,7,3
3,Bayview Hunters Point,2008-07-04 00:00:00,1,0,0.0,4.6,19.4,2008,7,4
4,Bayview Hunters Point,2008-07-05 00:00:00,0,0,0.0,4.6,19.4,2008,7,5


### **1. Mục tiêu:** Khu phố nào có nhu cầu bảo trì cây xanh cao nhất?

In [4]:
print("\n5 khu phố có nhu cầu bảo trì cây trung bình cao nhất")

top_nbhd_sql = pd.read_sql_query("""
    SELECT
        nbhd,
        COUNT(*) AS so_ngay,
        SUM(n) AS tong_so_ca,
        ROUND(AVG(n), 3) AS avg_n
    FROM demand
    GROUP BY nbhd
    ORDER BY avg_n DESC
    LIMIT 5;
""", conn)

top_nbhd_sql



5 khu phố có nhu cầu bảo trì cây trung bình cao nhất


,nbhd,so_ngay,tong_so_ca,avg_n
0,Mission,6664,19072,2.862
1,West of Twin Peaks,6664,10439,1.566
2,Bayview Hunters Point,6664,9019,1.353
3,Sunset/Parkside,6664,8191,1.229
4,Castro/Upper Market,6664,7938,1.191


**Nhận xét:** Top 5 khu phố có nhu cầu cao nhất

**Vì sao dùng truy vấn này?** Cột `nbhd` cho biết khu phố xảy ra yêu cầu bảo trì. Truy vấn dùng `GROUP BY nbhd` để tính trung bình số ca mỗi ngày (`avg_n`) cho từng khu phố, thay vì chỉ đếm tổng — vì các khu phố có số ngày dữ liệu khác nhau nên so sánh bằng trung bình mới công bằng.

**Kết quả cần đọc:** Mission dẫn đầu rõ rệt với trung bình 2.862 ca/ngày, gần gấp đôi khu phố đứng thứ 2 (West of Twin Peaks, 1.566) và gấp 2.4 lần khu phố thứ 5 (Castro/Upper Market, 1.191). Cả 5 khu phố đều có đủ 6,664 ngày dữ liệu như nhau, nên chênh lệch này phản ánh đúng sự khác biệt về nhu cầu thực tế giữa các khu phố, không phải do khác biệt về độ dài lịch sử dữ liệu. Mission là khu phố cần ưu tiên nguồn lực bảo trì cây xanh cao nhất trong 41 khu phố của San Francisco.

### **Mục tiêu:** Khu phố nào có nhu cầu bảo trì cây xanh thấp nhất?

In [5]:
print("\n5 khu phố có nhu cầu bảo trì cây trung bình thấp nhất")

bottom_nbhd_sql = pd.read_sql_query("""
    SELECT
        nbhd,
        COUNT(*) AS so_ngay,
        SUM(n) AS tong_so_ca,
        ROUND(AVG(n), 3) AS avg_n
    FROM demand
    GROUP BY nbhd
    ORDER BY avg_n ASC
    LIMIT 5;
""", conn)

bottom_nbhd_sql



5 khu phố có nhu cầu bảo trì cây trung bình thấp nhất


,nbhd,so_ngay,tong_so_ca,avg_n
0,Treasure Island,6664,101,0.015
1,Presidio,6664,115,0.017
2,Lincoln Park,6664,120,0.018
3,McLaren Park,6664,357,0.054
4,Seacliff,6664,641,0.096


**Nhận xét:** Top 5 khu phố có nhu cầu thấp nhất

**Vì sao dùng truy vấn này?** Cùng logic với truy vấn trước nhưng đổi `ORDER BY avg_n ASC` để lấy nhóm thấp nhất, dùng làm đối chứng cho mức chênh lệch giữa khu phố cao nhất và thấp nhất.

**Kết quả cần đọc:** Treasure Island có nhu cầu thấp nhất (0.015 ca/ngày, chỉ 101 ca trong suốt 6,664 ngày — trung bình hơn 2 tháng mới có 1 ca). So với Mission (2.862 ca/ngày) ở Query 1, mức chênh lệch lên tới **~191 lần** — cho thấy nhu cầu bảo trì cây xanh phân bố cực kỳ không đồng đều giữa các khu phố San Francisco. Đáng chú ý, 3/5 khu phố ở đây (Seacliff, Lincoln Park, Presidio) đều là những khu phố mà mô hình dự báo ở notebook 5 xác định là khó dự báo nhất khi kiểm thử trên khu phố chưa từng thấy — hợp lý vì nhu cầu quá thấp và thưa thớt khiến mô hình khó học được quy luật.

### **Mục tiêu:** Nhu cầu bảo trì cây thay đổi như thế nào theo thứ trong tuần?

In [6]:
print("\nThống kê nhu cầu bảo trì cây theo thứ trong tuần")

dow_sql = pd.read_sql_query("""
    SELECT
        dow,
        CASE dow
            WHEN 0 THEN 'Thứ Hai'
            WHEN 1 THEN 'Thứ Ba'
            WHEN 2 THEN 'Thứ Tư'
            WHEN 3 THEN 'Thứ Năm'
            WHEN 4 THEN 'Thứ Sáu'
            WHEN 5 THEN 'Thứ Bảy'
            WHEN 6 THEN 'Chủ Nhật'
        END AS thu_trong_tuan,
        SUM(n) AS tong_so_ca,
        ROUND(AVG(n), 3) AS avg_n
    FROM demand
    GROUP BY dow
    ORDER BY avg_n DESC;
""", conn)

dow_sql



Thống kê nhu cầu bảo trì cây theo thứ trong tuần


,dow,thu_trong_tuan,tong_so_ca,avg_n
0,1,Thứ Ba,28458,0.729
1,0,Thứ Hai,27575,0.706
2,2,Thứ Tư,27049,0.693
3,3,Thứ Năm,26882,0.689
4,4,Thứ Sáu,24466,0.627
5,6,Chủ Nhật,14976,0.384
6,5,Thứ Bảy,14639,0.375


**Nhận xét:** Phân bố nhu cầu theo thứ trong tuần

**Vì sao dùng truy vấn này?** Truy vấn dùng `GROUP BY dow` để tính nhu cầu trung bình cho từng thứ trong tuần. `CASE WHEN` chỉ dùng để đổi mã số (0-6) sang tên thứ dễ đọc, không ảnh hưởng đến kết quả tính toán.

**Kết quả cần đọc:** Có sự phân hóa rõ rệt giữa ngày làm việc và cuối tuần: Thứ Hai đến Thứ Sáu dao động 0.627–0.729 ca/ngày, trong khi Thứ Bảy và Chủ Nhật chỉ còn 0.375–0.384 — thấp hơn khoảng 45% so với ngày thường. Điều này cho thấy nhu cầu bảo trì cây có liên hệ chặt với hoạt động đô thị diễn ra chủ yếu vào ngày làm việc (giao thông, thi công, người dân đi lại phát hiện và báo cáo sự cố cây), chứ không phân bố ngẫu nhiên đều 7 ngày trong tuần. Đây là căn cứ để đội bảo trì ưu tiên bố trí nhân lực đông hơn vào các ngày trong tuần.

### **Mục tiêu:** Nhu cầu bảo trì cây thay đổi như thế nào theo mùa trong năm?

In [7]:
print("\nThống kê nhu cầu bảo trì cây theo mùa")

season_sql = pd.read_sql_query("""
    SELECT
        CASE
            WHEN month IN (12, 1, 2) THEN 'Đông'
            WHEN month IN (3, 4, 5) THEN 'Xuân'
            WHEN month IN (6, 7, 8) THEN 'Hè'
            ELSE 'Thu'
        END AS mua,
        SUM(n) AS tong_so_ca,
        ROUND(AVG(n), 3) AS avg_n
    FROM demand
    GROUP BY mua
    ORDER BY avg_n DESC;
""", conn)

season_sql



Thống kê nhu cầu bảo trì cây theo mùa


,mua,tong_so_ca,avg_n
0,Hè,46014,0.653
1,Xuân,40866,0.602
2,Thu,39939,0.585
3,Đông,37226,0.559


**Nhận xét:** Phân bố nhu cầu theo mùa

**Vì sao dùng truy vấn này?** Truy vấn gộp 12 tháng thành 4 mùa bằng `CASE WHEN month IN (...)`, sau đó tính nhu cầu trung bình mỗi mùa — giúp nhìn quy luật mùa vụ tổng quát hơn so với xem từng tháng riêng lẻ.

**Kết quả cần đọc:** Mùa Hè có nhu cầu cao nhất (0.653 ca/ngày), mùa Đông thấp nhất (0.559) — chênh lệch chỉ khoảng 17%, thấp hơn nhiều so với mức chênh theo khu phố (191 lần) hay theo ngày trong tuần (45%) ở các truy vấn trước. Đáng chú ý, mùa Hè (khô ráo, ít mưa ở San Francisco) lại có nhu cầu cao nhất chứ không phải mùa Đông (mùa mưa) — kết quả này đi ngược với giả thuyết ban đầu về ảnh hưởng của thời tiết cực đoan/bão, và củng cố thêm hướng giả thuyết rằng hoạt động đô thị/lịch cắt tỉa định kỳ có vai trò lớn hơn thời tiết trong việc quyết định nhu cầu bảo trì — cần kiểm định thêm bằng tương quan Spearman ở notebook EDA.

### **Mục tiêu:** Nhu cầu bảo trì cây phân bố như thế nào theo từng tháng?

In [8]:
print("\nThống kê nhu cầu bảo trì cây theo từng tháng")

month_sql = pd.read_sql_query("""
    SELECT
        month,
        SUM(n) AS tong_so_ca,
        ROUND(AVG(n), 3) AS avg_n
    FROM demand
    GROUP BY month
    ORDER BY avg_n DESC;
""", conn)

month_sql



Thống kê nhu cầu bảo trì cây theo từng tháng


,month,tong_so_ca,avg_n
0,6,14993,0.677
1,10,14718,0.643
2,8,15536,0.643
3,7,15485,0.641
4,5,14384,0.629
5,2,12758,0.613
6,9,14081,0.605
7,3,13591,0.594
8,1,13492,0.590
9,4,12891,0.582


**Nhận xét:** Phân bố nhu cầu theo tháng

**Vì sao dùng truy vấn này?** Truy vấn dùng `GROUP BY month` để xem chi tiết hơn truy vấn theo mùa ở trên — mỗi mùa có 3 tháng nhưng có thể không đồng đều với nhau.

**Kết quả cần đọc:** Tháng 6 có nhu cầu cao nhất (0.677), khớp với mùa Hè dẫn đầu ở Query 4. Đáng chú ý nhất là tháng 11 (0.503) và tháng 12 (0.480) thấp rõ rệt so với cả 2 tháng mùa Đông còn lại (tháng 1: 0.590, tháng 2: 0.613) — cho thấy mức thấp của mùa Đông không đến từ cả 3 tháng đều nhau, mà chủ yếu do một điểm trũng riêng vào cuối năm, trùng với giai đoạn lễ Tạ Ơn và Giáng Sinh — có thể do người dân ít báo cáo hơn hoặc năng lực xử lý 311 giảm trong kỳ nghỉ lễ. Tổng theo tháng khớp tuyệt đối với tổng theo mùa ở Query 4, xác nhận logic gộp nhóm nhất quán giữa 2 truy vấn.

### **Mục tiêu:** Nhu cầu bảo trì cây thay đổi như thế nào qua các năm?

In [9]:
print("\nThống kê nhu cầu bảo trì cây theo từng năm")

year_sql = pd.read_sql_query("""
    SELECT
        year,
        SUM(n) AS tong_so_ca,
        ROUND(AVG(n), 3) AS avg_n
    FROM demand
    GROUP BY year
    ORDER BY year;
""", conn)

year_sql



Thống kê nhu cầu bảo trì cây theo từng năm


,year,tong_so_ca,avg_n
0,2008,2226,0.295
1,2009,4624,0.309
2,2010,4104,0.274
3,2011,4606,0.308
4,2012,5305,0.354
5,2013,5450,0.364
6,2014,6119,0.409
7,2015,7180,0.480
8,2016,7961,0.531
9,2017,11146,0.745


**Nhận xét:** Xu hướng nhu cầu theo năm

**Vì sao dùng truy vấn này?** Truy vấn dùng `GROUP BY year` để cộng dồn và tính trung bình nhu cầu từng năm, dùng để phát hiện xu hướng dài hạn: tăng, giảm hay ổn định.

**Kết quả cần đọc:** Nhu cầu tăng rõ rệt và khá đều đặn qua 18 năm, từ 0.295 ca/ngày (2008) lên 1.136 (2026) — tăng gần 3.85 lần, đặc biệt tăng nhanh giai đoạn 2014-2019. Có 2 điểm bất thường đáng chú ý: (1) năm 2020 giảm còn 0.612 so với 2019 (0.767), trùng đúng giai đoạn phong tỏa COVID-19 khiến ít người đi lại phát hiện/báo cáo cây hỏng hơn; (2) năm 2022 giảm mạnh còn 0.573 rồi phục hồi ngay năm

### **Mục tiêu:** Tổ hợp khu phố và thứ trong tuần nào có nhu cầu cao nhất?

In [10]:
print("\nTổ hợp khu phố và thứ trong tuần có nhu cầu bảo trì cao nhất")

nbhd_dow_hotspot_sql = pd.read_sql_query("""
    SELECT
        nbhd,
        dow,
        ROUND(AVG(n), 3) AS avg_n,
        SUM(n) AS tong_so_ca
    FROM demand
    GROUP BY nbhd, dow
    ORDER BY avg_n DESC
    LIMIT 10;
""", conn)

nbhd_dow_hotspot_sql



Tổ hợp khu phố và thứ trong tuần có nhu cầu bảo trì cao nhất


,nbhd,dow,avg_n,tong_so_ca
0,Mission,1,3.453,3287
1,Mission,3,3.404,3241
2,Mission,0,3.209,3055
3,Mission,2,3.119,2969
4,Mission,4,2.922,2782
5,Mission,5,1.988,1893
6,Mission,6,1.938,1845
7,West of Twin Peaks,1,1.828,1740
8,West of Twin Peaks,0,1.820,1733
9,Bayview Hunters Point,1,1.815,1728


**Nhận xét:** Điểm nóng theo tổ hợp khu phố và thứ trong tuần

**Vì sao dùng truy vấn này?** Truy vấn nhóm đồng thời theo `nbhd` và `dow` — đây là phép tổng hợp hai chiều để tìm tổ hợp không gian-thời gian có nhu cầu cao nhất, thay vì chỉ xét riêng từng chiều.

**Kết quả cần đọc:** Mission chiếm trọn 7/10 vị trí đầu bảng — nghĩa là cả 7 ngày trong tuần của riêng Mission đều cao hơn ngày cao nhất của bất kỳ khu phố nào khác. Điều này khẳng định nhu cầu cao chủ yếu do đặc trưng không gian (khu phố) quyết định, còn yếu tố thời gian (thứ trong tuần) chỉ đóng vai trò điều chỉnh nhẹ bên trong mỗi khu phố. Bên trong Mission, thứ tự các ngày (Thứ Ba > Thứ Năm > Thứ Hai > Thứ Tư > Thứ Sáu > Thứ Bảy > Chủ Nhật) khớp đúng xu hướng ngày-làm-việc-cao-hơn-cuối-tuần đã thấy ở Query 3 trên toàn thành phố. Tổng 7 dòng của Mission (19,072 ca) khớp tuyệt đối với tổng số ca của Mission ở Query 1, xác nhận tính nhất quán giữa các truy vấn.

### **Mục tiêu:** Khu phố nào vượt trội hơn bao nhiêu lần so với mức trung bình toàn thành phố?

In [11]:
print("\nSo sánh nhu cầu từng khu phố với trung bình toàn thành phố")

city_compare_sql = pd.read_sql_query("""
    WITH city_avg AS (
        SELECT AVG(n) AS avg_city FROM demand
    ),
    nbhd_avg AS (
        SELECT nbhd, AVG(n) AS avg_nbhd
        FROM demand
        GROUP BY nbhd
    )
    SELECT
        nbhd_avg.nbhd,
        ROUND(nbhd_avg.avg_nbhd, 3) AS avg_nbhd,
        ROUND(city_avg.avg_city, 3) AS avg_city,
        ROUND(nbhd_avg.avg_nbhd - city_avg.avg_city, 3) AS chenh_lech,
        ROUND(nbhd_avg.avg_nbhd * 1.0 / city_avg.avg_city, 2) AS lan_so_voi_tb
    FROM nbhd_avg, city_avg
    ORDER BY avg_nbhd DESC
    LIMIT 5;
""", conn)

city_compare_sql



So sánh nhu cầu từng khu phố với trung bình toàn thành phố


,nbhd,avg_nbhd,avg_city,chenh_lech,lan_so_voi_tb
0,Mission,2.862,0.6,2.262,4.77
1,West of Twin Peaks,1.566,0.6,0.966,2.61
2,Bayview Hunters Point,1.353,0.6,0.753,2.25
3,Sunset/Parkside,1.229,0.6,0.629,2.05
4,Castro/Upper Market,1.191,0.6,0.591,1.98


**Nhận xét:** So sánh với trung bình thành phố bằng CTE

**Vì sao dùng truy vấn này?** Truy vấn dùng CTE (`WITH ... AS`) để tách 2 bước tính toán: `city_avg` tính trung bình chung toàn thành phố, `nbhd_avg` tính trung bình từng khu phố — sau đó ghép lại để tính độ lệch (`chenh_lech`) và tỷ lệ (`lan_so_voi_tb`). Cách viết bằng CTE giúp câu lệnh rõ ràng, dễ đọc hơn so với lồng subquery trực tiếp.

**Kết quả cần đọc:** Mission cao gấp **4.77 lần** mức trung bình toàn thành phố (0.6 ca/ngày) — mức chênh lệch rất lớn, khẳng định lại phát hiện ở Query 1-2 bằng một thước đo khác (tỷ lệ so với trung bình chung thay vì so sánh trực tiếp giữa các khu phố). Ngay cả khu phố đứng thứ 5 (Castro/Upper Market) vẫn gần gấp đôi (1.98 lần) mức trung bình thành phố — cho thấy

### **Mục tiêu:** Khu phố nào đứng đầu về nhu cầu bảo trì trong mỗi tháng?

In [12]:
print("\nKhu phố có nhu cầu bảo trì cao nhất trong từng tháng")

top_per_month_sql = pd.read_sql_query("""
    WITH nbhd_month AS (
        SELECT
            nbhd,
            month,
            AVG(n) AS avg_n
        FROM demand
        GROUP BY nbhd, month
    ),
    ranked AS (
        SELECT
            *,
            ROW_NUMBER() OVER (PARTITION BY month ORDER BY avg_n DESC) AS hang
        FROM nbhd_month
    )
    SELECT
        month,
        nbhd,
        ROUND(avg_n, 3) AS avg_n
    FROM ranked
    WHERE hang = 1
    ORDER BY month;
""", conn)

top_per_month_sql



Khu phố có nhu cầu bảo trì cao nhất trong từng tháng


,month,nbhd,avg_n
0,1,Mission,2.805
1,2,Mission,2.695
2,3,Mission,2.808
3,4,Mission,2.672
4,5,Mission,3.174
5,6,Mission,3.230
6,7,Mission,2.969
7,8,Mission,3.036
8,9,Mission,2.986
9,10,Mission,3.188


**Nhận xét:** Khu phố dẫn đầu mỗi tháng

**Vì sao dùng truy vấn này?** Truy vấn dùng cửa sổ trượt `ROW_NUMBER() OVER (PARTITION BY month ORDER BY avg_n DESC)` để xếp hạng các khu phố trong từng tháng riêng biệt, sau đó lọc `hang = 1` để chỉ giữ khu phố đứng đầu mỗi tháng.

**Kết quả cần đọc:** Mission đứng đầu cả 12/12 tháng trong năm, không có tháng nào bị khu phố khác vượt qua — khẳng định vị trí điểm nóng số 1 của Mission là ổn định tuyệt đối, hoàn toàn không phụ thuộc vào mùa vụ. Đáng chú ý, ngay cả trong nội bộ Mission, tháng 11 (2.280) và tháng 12 (2.455) vẫn là 2 tháng thấp nhất trong năm — khớp đúng với điểm trũng cuối năm đã phát hiện ở Query 5, cho thấy đây là hiện tượng chung ảnh hưởng đồng đều lên toàn thành phố, kể cả khu phố có nhu cầu cao nhất, chứ không phải đặc thù riêng

### **Mục tiêu:** Nhu cầu bảo trì cây có khác nhau giữa các mức hoạt động thi công đường phố không?

In [13]:
print("\nNhu cầu bảo trì cây theo mức độ hoạt động thi công đường phố (permits_active)")
print("Chỉ tính từ 2015 trở đi vì permits_active chưa có dữ liệu thật trước năm này")

permits_group_sql = pd.read_sql_query("""
    SELECT
        CASE
            WHEN permits_active = 0 THEN '1. Không có giấy phép'
            WHEN permits_active <= 12 THEN '2. Thấp (1-12)'
            WHEN permits_active <= 65 THEN '3. Trung bình (13-65)'
            ELSE '4. Cao (>65)'
        END AS nhom_permits,
        COUNT(*) AS so_dong,
        ROUND(AVG(n), 3) AS avg_n
    FROM demand
    WHERE date >= '2015-01-01'
    GROUP BY nhom_permits
    ORDER BY nhom_permits;
""", conn)

permits_group_sql



Nhu cầu bảo trì cây theo mức độ hoạt động thi công đường phố (permits_active)
Chỉ tính từ 2015 trở đi vì permits_active chưa có dữ liệu thật trước năm này


,nhom_permits,so_dong,avg_n
0,1. Không có giấy phép,3372,0.029
1,2. Thấp (1-12),32395,0.125
2,3. Trung bình (13-65),61395,0.646
3,4. Cao (>65),78687,1.116


**Nhận xét:** Nhu cầu theo mức độ hoạt động thi công đường phố

**Vì sao dùng truy vấn này?** Truy vấn chia `permits_active` thành 4 nhóm bằng `CASE WHEN` (ngưỡng dựa theo tứ phân vị đã quan sát ở Notebook 2: median 12, phân vị 75 là 65), rồi so sánh `avg_n` giữa các nhóm. Bắt buộc lọc `WHERE date >= '2015-01-01'` vì trước năm này `permits_active` luôn bằng 0 do thiếu dữ liệu nguồn phụ, đưa vào sẽ làm sai lệch kết quả so sánh.

**Kết quả cần đọc:** Xu hướng tăng rất mạnh và đơn điệu: từ 0.029 ca/ngày (không có giấy phép) lên 1.116 (permits_active > 65) — tăng gần 38 lần. Tuy nhiên, đây chỉ là bằng chứng ở mức mô tả (descriptive), cần diễn giải thận trọng: mức tăng khổng lồ này nhiều khả năng bị **gây nhiễu bởi chính đặc trưng khu phố** — các khu phố sầm uất như Mission vừa có nhiều giấy phép thi công vừa có nhiều case cây, nên bảng này có thể đang đo lại sự khác biệt "khu phố bận rộn vs khu phố yên tĩnh" hơn là quan hệ nhân quả thật giữa thi công và nhu cầu cây trong cùng một khu phố. Đây chính là cơ sở thực nghiệm cho nghịch lý ở RQ4: SHAP xếp permits_active khá quan trọng nhưng kiểm định ablation ở

### **Mục tiêu:** Ngày có mưa và ngày không mưa có khác biệt về nhu cầu bảo trì cây không?

In [14]:
print("\nNhu cầu bảo trì cây vào ngày có mưa và không mưa")
print("Chỉ tính từ 2015 trở đi vì PRCP chưa có dữ liệu thật trước năm này")

rain_sql = pd.read_sql_query("""
    SELECT
        CASE WHEN PRCP > 0 THEN 'Có mưa' ELSE 'Không mưa' END AS tinh_trang_mua,
        COUNT(*) AS so_dong,
        ROUND(AVG(n), 3) AS avg_n
    FROM demand
    WHERE date >= '2015-01-01'
    GROUP BY tinh_trang_mua;
""", conn)

rain_sql



Nhu cầu bảo trì cây vào ngày có mưa và không mưa
Chỉ tính từ 2015 trở đi vì PRCP chưa có dữ liệu thật trước năm này


,tinh_trang_mua,so_dong,avg_n
0,Có mưa,27634,0.882
1,Không mưa,148215,0.723


**Nhận xét:** So sánh nhu cầu giữa ngày mưa và không mưa

**Vì sao dùng truy vấn này?** Truy vấn tách 2 nhóm ngày dựa trên `PRCP > 0`, so sánh `avg_n` giữa 2 nhóm — cách làm đơn giản để có cái nhìn đầu tiên về ảnh hưởng của mưa trước khi đi sâu phân tích tương quan ở notebook EDA.

**Kết quả cần đọc:** Ngày có mưa có nhu cầu trung bình cao hơn khoảng 22% so với ngày không mưa (0.882 so với 0.723). Thoạt nhìn có vẻ mâu thuẫn với hệ số tương quan Spearman rất yếu giữa PRCP và nhu cầu (rho=0.014) đã phát hiện ở bài tập mở rộng trước đó — nhưng thực chất không mâu thuẫn: trung bình (mean) rất nhạy với các ngày mưa lớn gây đột biến case (do target vốn lệch phải mạnh), trong khi Spearman đo theo thứ hạng nên không bị chi phối bởi vài ngày ngoại lệ. Điều này gợi ý tác động của mưa mang tính "ngưỡng" (có mưa hay không) hơn là tuyến tính theo lượng mưa (mưa nhiều gấp đôi không có nghĩa case tăng gấp đôi) — một sắc thái quan trọng cần làm rõ ở phần EDA/tương quan tiếp theo.

### **Mục tiêu:** Nhu cầu bảo trì cây phân bố như thế nào theo phân vị (decile) toàn thành phố?

In [15]:
print("\nPhân phối số ca bảo trì theo 10 nhóm phân vị (decile)")

decile_sql = pd.read_sql_query("""
    WITH ranked AS (
        SELECT
            n,
            NTILE(10) OVER (ORDER BY n) AS decile
        FROM demand
    )
    SELECT
        decile,
        COUNT(*) AS so_dong,
        MIN(n) AS n_min,
        MAX(n) AS n_max,
        ROUND(AVG(n), 3) AS avg_n
    FROM ranked
    GROUP BY decile
    ORDER BY decile;
""", conn)

decile_sql



Phân phối số ca bảo trì theo 10 nhóm phân vị (decile)


,decile,so_dong,n_min,n_max,avg_n
0,1,27323,0,0,0.000
1,2,27323,0,0,0.000
2,3,27323,0,0,0.000
3,4,27323,0,0,0.000
4,5,27322,0,0,0.000
5,6,27322,0,0,0.000
6,7,27322,0,1,0.236
7,8,27322,1,1,1.000
8,9,27322,1,2,1.323
9,10,27322,2,86,3.446


**Nhận xét:** Phân phối theo phân vị (decile)

**Vì sao dùng truy vấn này?**
Truy vấn dùng `NTILE(10) OVER (ORDER BY n)` để chia toàn bộ 273,224 dòng thành 10 nhóm bằng nhau theo giá trị `n` tăng dần (decile 1 = 10% dòng có `n` thấp nhất, decile 10 = 10% dòng có `n` cao nhất). Cách này cho cái nhìn về độ lệch phân phối rõ hơn nhiều so với chỉ xem trung bình/trung vị.

**Kết quả cần đọc:** Phát hiện gây bất ngờ: 6/10 decile (tương đương 60% dữ liệu) đều có `n_max = 0` — nghĩa là hơn một nửa số dòng khu phố-ngày trong suốt lịch sử hoàn toàn không phát sinh ca bảo trì nào. Chỉ từ decile 7 trở lên mới bắt đầu có giá trị dương, và decile 10 (10% cao nhất) mới thực sự tập trung phần lớn nhu cầu (avg_n=3.446, max=86) — bước nhảy từ decile 9 (avg 1.323, max 2) sang decile 10 lớn hơn hẳn các bước nhảy trước, xác nhận phân phối lệch phải cực mạnh. Điều này giải thích tại sao bài toán dự báo khó hơn con số MAE tuyệt đối cho thấy: phần lớn ngày dễ đoán (bằng 0), còn phần khó — các đợt tăng đột biến hiếm hoi ở decile 10 — mới là nơi mô hình thực sự bị thử thách, và cũng lý giải vì sao recall cảnh báo sự kiện ở RQ3 chỉ đạt mức khiêm tốn (0.26-0.375).

### **Mục tiêu:** Những ngày nào trong lịch sử có tổng nhu cầu bảo trì cây cao nhất toàn thành phố?

In [16]:
print("\nTop 10 ngày có tổng số ca bảo trì cây cao nhất (toàn thành phố)")

top_days_sql = pd.read_sql_query("""
    SELECT
        date,
        SUM(n) AS tong_so_ca_toan_tp
    FROM demand
    GROUP BY date
    ORDER BY tong_so_ca_toan_tp DESC
    LIMIT 10;
""", conn)

top_days_sql



Top 10 ngày có tổng số ca bảo trì cây cao nhất (toàn thành phố)


,date,tong_so_ca_toan_tp
0,2021-10-24 00:00:00,553
1,2023-03-21 00:00:00,479
2,2021-10-25 00:00:00,424
3,2024-02-04 00:00:00,410
4,2024-02-05 00:00:00,330
5,2021-01-19 00:00:00,301
6,2023-03-22 00:00:00,294
7,2024-12-14 00:00:00,271
8,2017-01-08 00:00:00,271
9,2009-10-13 00:00:00,205


**Nhận xét:** Các ngày cao điểm nhất trong lịch sử

**Vì sao dùng truy vấn này?** Truy vấn gộp toàn bộ 41 khu phố lại theo `GROUP BY date` để tìm những ngày mà cả thành phố cùng lúc có nhu cầu bảo trì tăng vọt — khác với các truy vấn trước chỉ xem trung bình dài hạn, truy vấn này tìm đúng các sự kiện đơn lẻ bất thường.

**Kết quả cần đọc:** Đã xác minh bằng tra cứu thực tế: 4/5 vị trí đầu bảng trùng khớp chính xác với các sự kiện bão "atmospheric river" nổi tiếng tại California. Ngày 24/10/2021 (#1, 553 ca) và 25/10/2021 (#3, 424 ca) là "bomb cyclone" lịch sử — San Francisco ghi nhận lượng mưa cao thứ 4 trong lịch sử đo đạc. Ngày 4/2/2024 (#4, 410 ca) và 5/2/2024 (#5, 330 ca) là đợt atmospheric river gây thiệt hại ~3 tỷ USD toàn California, báo chí đưa tin trực tiếp về cây đổ chắn đường tại phố Fulton và Fillmore ở San Francisco đúng ngày này. Đây là bằng chứng thực tế mạnh mẽ: dù tương quan trung bình dài hạn giữa thời tiết và nhu cầu khá yếu (Query 11, bài tập mở rộng), các sự kiện thời tiết cực đoan riêng lẻ vẫn gây ra đột biến rõ rệt — khẳng định notebook 5 cần có cơ chế phát hiện/cảnh báo sự kiện riêng biệt (event detection) thay vì chỉ dựa vào tương quan trung bình.

### **Mục tiêu:** Tổ hợp khu phố và mùa nào có nhu cầu bảo trì cây cao nhất?

In [17]:
print("\nTổ hợp khu phố và mùa có nhu cầu bảo trì cây cao nhất")

nbhd_season_sql = pd.read_sql_query("""
    SELECT
        nbhd,
        CASE
            WHEN month IN (12, 1, 2) THEN 'Đông'
            WHEN month IN (3, 4, 5) THEN 'Xuân'
            WHEN month IN (6, 7, 8) THEN 'Hè'
            ELSE 'Thu'
        END AS mua,
        ROUND(AVG(n), 3) AS avg_n,
        SUM(n) AS tong_so_ca
    FROM demand
    GROUP BY nbhd, mua
    ORDER BY avg_n DESC
    LIMIT 10;
""", conn)

nbhd_season_sql



Tổ hợp khu phố và mùa có nhu cầu bảo trì cây cao nhất


,nbhd,mua,avg_n,tong_so_ca
0,Mission,Hè,3.074,5281
1,Mission,Xuân,2.887,4781
2,Mission,Thu,2.825,4706
3,Mission,Đông,2.650,4304
4,West of Twin Peaks,Đông,1.608,2612
5,West of Twin Peaks,Thu,1.578,2629
6,West of Twin Peaks,Xuân,1.566,2593
7,West of Twin Peaks,Hè,1.516,2605
8,Bayview Hunters Point,Hè,1.449,2489
9,Bayview Hunters Point,Đông,1.406,2283


**Nhận xét:** Điểm nóng theo tổ hợp khu phố và mùa

**Vì sao dùng truy vấn này?** Tương tự truy vấn tổ hợp khu phố × thứ trong tuần ở trên, nhưng đổi chiều thời gian sang mùa — giúp kiểm tra xem khu phố có nhu cầu cao nhất có bị ảnh hưởng thêm bởi mùa vụ hay giữ nguyên thứ hạng quanh năm.

**Kết quả cần đọc:** Mission và West of Twin Peaks chiếm 8/10 vị trí đầu bảng (4 mùa mỗi khu phố), khẳng định lại kết luận ở Query 7-9: nhu cầu cao chủ yếu do đặc trưng khu phố quyết định. Tuy nhiên, phát hiện bất ngờ: **2 khu phố này có xu hướng mùa vụ gần như ngược nhau**. Mission có mùa cao điểm là Hè (3.074) > Xuân > Thu > Đông (2.650), khớp đúng xu hướng chung toàn thành phố ở Query 4. Ngược lại, West of Twin Peaks lại cao điểm vào Đông (1.608) > Thu > Xuân > Hè (1.516) — gần như đảo ngược hoàn toàn. Điều này cho thấy mùa vụ không ảnh hưởng đồng nhất lên toàn thành phố mà tác động khác nhau tùy đặc điểm địa lý từng khu phố (West of Twin Peaks nằm trên địa hình đồi cao, nhiều công viên/cây xanh, có thể chịu ảnh hưởng trực tiếp của gió mùa Đông hơn khu đô thị dày đặc như Mission) — một phát hiện quan trọng cho việc thiết kế mô hình dự báo riêng biệt theo đặc trưng khu phố ở notebook 5, thay vì áp dụng chung một quy luật mùa vụ cho toàn thành phố.

### **Mục tiêu:** Khu phố nào có tỷ lệ ngày "không có ca nào" cao nhất/thấp nhất?

In [18]:
print("\nTỷ lệ ngày không phát sinh ca bảo trì nào (n = 0) theo từng khu phố")

zero_days_sql = pd.read_sql_query("""
    SELECT
        nbhd,
        COUNT(*) AS tong_so_ngay,
        SUM(CASE WHEN n = 0 THEN 1 ELSE 0 END) AS so_ngay_khong_co_ca,
        ROUND(
            100.0 * SUM(CASE WHEN n = 0 THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS ty_le_phan_tram
    FROM demand
    GROUP BY nbhd
    ORDER BY ty_le_phan_tram DESC
    LIMIT 5;
""", conn)

zero_days_sql



Tỷ lệ ngày không phát sinh ca bảo trì nào (n = 0) theo từng khu phố


,nbhd,tong_so_ngay,so_ngay_khong_co_ca,ty_le_phan_tram
0,Treasure Island,6664,6596,98.98
1,Presidio,6664,6554,98.35
2,Lincoln Park,6664,6551,98.30
3,McLaren Park,6664,6355,95.36
4,Seacliff,6664,6109,91.67


**Nhận xét:** Tỷ lệ ngày "yên tĩnh" (không có ca) theo khu phố

**Vì sao dùng truy vấn này?** Truy vấn dùng `SUM(CASE WHEN n = 0 THEN 1 ELSE 0 END)` để đếm số ngày mà khu phố đó hoàn toàn không có yêu cầu bảo trì nào, sau đó tính tỷ lệ phần trăm — góc nhìn bổ sung cho Query 1-2 vốn chỉ nhìn giá trị trung bình.

**Kết quả cần đọc:** Top 5 khu phố có tỷ lệ ngày trống cao nhất (90.98%–98.98%) trùng khớp chính xác về thứ tự với Top 5 khu phố có nhu cầu thấp nhất ở Query 2 — cho thấy ở nhóm cực trị thấp, 2 chỉ số này đi liền nhau chứ không tách biệt như giả thuyết ban đầu. Tuy vậy, khi tính thêm "trung bình trên những ngày có phát sinh case" (lấy tổng số ca ở Query 2 chia cho số ngày hoạt động), Treasure Island vẫn nổi bật là khu "dồn cục" nhất (≈1.49 ca mỗi lần phát sinh) so với Presidio gần như luôn đúng 1 ca mỗi lần (≈1.05) — một khác biệt tinh tế về bản chất "bùng phát" dù cả hai đều có nhu cầu tổng thể rất thấp và hiếm.

In [19]:
sql_results = {
    "sql_01_top_nbhd.csv": top_nbhd_sql,
    "sql_02_bottom_nbhd.csv": bottom_nbhd_sql,
    "sql_03_dow.csv": dow_sql,
    "sql_04_season.csv": season_sql,
    "sql_05_month.csv": month_sql,
    "sql_06_year.csv": year_sql,
    "sql_07_nbhd_dow_hotspot.csv": nbhd_dow_hotspot_sql,
    "sql_08_city_compare.csv": city_compare_sql,
    "sql_09_top_per_month.csv": top_per_month_sql,
    "sql_10_permits_group.csv": permits_group_sql,
    "sql_11_rain.csv": rain_sql,
    "sql_12_decile.csv": decile_sql,
    "sql_13_top_days.csv": top_days_sql,
    "sql_14_nbhd_season.csv": nbhd_season_sql,
    "sql_15_zero_days.csv": zero_days_sql
}

for filename, result_df in sql_results.items():
    result_df.to_csv(filename, index=False)

print("Đã lưu xong 15 bảng kết quả SQL.")


Đã lưu xong 15 bảng kết quả SQL.


In [20]:
conn.close()
print("\nĐã đóng kết nối SQLite.")



Đã đóng kết nối SQLite.
